In [31]:
import pandas as pd 
import numpy as np 


In [32]:
df_raw = pd.read_csv("../data/raw/Cricket.csv")

In [33]:
print("Raw cricket dataset-Initial check\n")
print(f"Dataset shape: {df_raw.shape[0]} rows * {df_raw.shape[1]} coloums")
print("*"*60)
print("Coloums Names and Data types")
print(df_raw.dtypes)
print("*"*60)

print(f"\nFirst 3 rows:")
print("*"*60)

print(df_raw.head(3))

Raw cricket dataset-Initial check

Dataset shape: 1224 rows * 9 coloums
************************************************************
Coloums Names and Data types
name              object
main_category     object
sub_category      object
image             object
link              object
ratings           object
no_of_ratings     object
discount_price    object
actual_price      object
dtype: object
************************************************************

First 3 rows:
************************************************************
                                                name     main_category  \
0  Tusker Cricket Bat Toe Protector, Self Adhesiv...  sports & fitness   
1  Amazon Brand - Symactive Heavy Tennis Cricket ...  sports & fitness   
2  Lycan Safety Anti Crack Water Proof Cricket Ba...  sports & fitness   

  sub_category                                              image  \
0      Cricket  https://m.media-amazon.com/images/W/IMAGERENDE...   
1      Cricket  https://m.m

In [34]:
# replace 'Price' with your actual column name
df_raw["actual_price"] = (
    df_raw["actual_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_raw["actual_price"] = pd.to_numeric(df_raw["actual_price"], errors="coerce")

In [35]:
print("Cleaned actual_price column:")
print(df_raw["actual_price"].head)
print("\n" + "="*60)
print(f"Data type: {df_raw['actual_price'].dtype}")
print(f"Min price: {df_raw['actual_price'].min()}")
print(f"Max price: {df_raw['actual_price'].max()}")
print(f"Missing values: {df_raw['actual_price'].isna().sum()}")

Cleaned actual_price column:
<bound method NDFrame.head of 0        399.0
1        579.0
2        449.0
3        699.0
4       1249.0
         ...  
1219    1399.0
1220    1999.0
1221     599.0
1222    1999.0
1223     899.0
Name: actual_price, Length: 1224, dtype: float64>

Data type: float64
Min price: 90.0
Max price: 49499.0
Missing values: 34


In [36]:
# 1. Clean discount_price column (same as actual_price)
df_raw["discount_price"] = (
    df_raw["discount_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_raw["discount_price"] = pd.to_numeric(df_raw["discount_price"], errors="coerce")

In [37]:
# 2. Convert ratings to numeric
df_raw["ratings"] = pd.to_numeric(df_raw["ratings"], errors="coerce")

In [38]:
# 3. Clean no_of_ratings column (remove commas)
df_raw["no_of_ratings"] = (
    df_raw["no_of_ratings"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_raw["no_of_ratings"] = pd.to_numeric(df_raw["no_of_ratings"], errors="coerce")

In [39]:
# 4. Remove leading/trailing whitespace from text columns
df_raw["name"] = df_raw["name"].str.strip()
df_raw["main_category"] = df_raw["main_category"].str.strip()
df_raw["sub_category"] = df_raw["sub_category"].str.strip()

In [40]:
# 5. Handle missing values - check which columns have missing data
print("Missing values summary:")
print(df_raw.isnull().sum())
print("\n" + "="*60)
print(f"Total rows with any missing value: {df_raw.isnull().any(axis=1).sum()}")

Missing values summary:
name                0
main_category       0
sub_category        0
image               0
link                0
ratings           326
no_of_ratings     326
discount_price    160
actual_price       34
dtype: int64

Total rows with any missing value: 456


In [41]:
#creatinf discount percentage coloumn 
df_raw["discount_percentage"] = (
    ((df_raw["actual_price"] - df_raw["discount_price"]) / df_raw["actual_price"]) * 100
).round(2)

In [42]:
# removing duplicated data 
print(f"Duplicate rows before removal: {df_raw.duplicated().sum()}")
df_raw = df_raw.drop_duplicates()
print(f"Duplicate rows after removal: {df_raw.duplicated().sum()}")
print(f"Final dataset shape: {df_raw.shape}")

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Final dataset shape: (1224, 10)


In [43]:
#outliers removing from price columns through IQR method

Q1_actual = df_raw["actual_price"].quantile(0.25)
Q3_actual = df_raw["actual_price"].quantile(0.75)
IQR_actual = Q3_actual-Q1_actual

print(f"Actual Price - Q1: {Q1_actual}, Q3: {Q3_actual}, IQR: {IQR_actual}")
print(f"Lower bound: {Q1_actual-1.5*IQR_actual}")
print(f"Upper bound: {Q3_actual+1.5*IQR_actual}")

outliers = df_raw[
    (df_raw["actual_price"] < Q1_actual - 1.5*IQR_actual) | 
    (df_raw["actual_price"] > Q3_actual + 1.5*IQR_actual)
]
print(f"\nPotential outliers: {len(outliers)} rows")
# Uncomment below to remove outliers
df_raw = df_raw[~((df_raw["actual_price"] < Q1_actual - 1.5*IQR_actual) | (df_raw["actual_price"] > Q3_actual + 1.5*IQR_actual))]

Actual Price - Q1: 995.75, Q3: 2299.0, IQR: 1303.25
Lower bound: -959.125
Upper bound: 4253.875

Potential outliers: 72 rows


In [44]:
#cleaned dataset summary

print("="*60)
print("CLEANED DATASET SUMMARY")
print("="*60)
print(f"\nShape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names and types:")
print(df_raw.dtypes)
print(f"\nBasic statistics:")
print(df_raw.describe())
print(f"\nSample cleaned data:")
print(df_raw.head())

CLEANED DATASET SUMMARY

Shape: 1152 rows × 10 columns

Column names and types:
name                    object
main_category           object
sub_category            object
image                   object
link                    object
ratings                float64
no_of_ratings          float64
discount_price         float64
actual_price           float64
discount_percentage    float64
dtype: object

Basic statistics:
          ratings  no_of_ratings  discount_price  actual_price  \
count  846.000000     846.000000     1004.000000   1118.000000   
mean     3.768558     102.072104      837.232251   1606.710438   
std      0.782216     690.675266      600.007836    891.163927   
min      1.000000       1.000000       94.000000     90.000000   
25%      3.400000       3.000000      429.000000    929.000000   
50%      3.800000      11.000000      650.000000   1499.500000   
75%      4.200000      42.750000      999.000000   2041.000000   
max      5.000000   16635.000000     3649.000000 

In [45]:

import os
os.makedirs("../data/cleaned", exist_ok=True)
df_raw.to_csv("../data/cleaned/cricket.csv", index=False)
# print("Saved: ../data/cleaned/cricket.csv")